exploração dos dados

## configurações

In [1]:
import sys
import os
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as img
import random

import urllib.request

import tarfile

In [2]:
# Garante que a raiz do projeto está no sys.path para importar src/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(f"Raiz do projeto: {ROOT}")

Raiz do projeto: /home/carolina/Documents/Triagem_de_Raio-X


In [ ]:
# ── Diretórios ───────────────────────────────────────────────────────
PASTA_RAW_DATA   = Path(ROOT, "data", "raw")
PASTA_RAW_DATA.mkdir(parents=True, exist_ok=True)

PASTA_INPUT_DATA   = Path(ROOT, "data", "input")
PASTA_INPUT_DATA.mkdir(parents=True, exist_ok=True)

Dados blutos serão salvos em: /home/carolina/Documents/Triagem_de_Raio-X/data/raw


In [9]:
data = os.listdir(PASTA_RAW_DATA)
path_df = [i for i in data if i.endswith(".csv")]
df = pd.read_csv(PASTA_RAW_DATA/path_df[0])
df.head(10)

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Sex,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
0,00000001_000.png,Cardiomegaly,0,1,57,M,PA,2682,2749,0.143,0.143
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168
3,00000002_000.png,No Finding,0,2,80,M,PA,2500,2048,0.171,0.171
4,00000003_001.png,Hernia,0,3,74,F,PA,2500,2048,0.168,0.168
5,00000003_002.png,Hernia,1,3,75,F,PA,2048,2500,0.168,0.168
6,00000003_003.png,Hernia|Infiltration,2,3,76,F,PA,2698,2991,0.143,0.143
7,00000003_004.png,Hernia,3,3,77,F,PA,2500,2048,0.168,0.168
8,00000003_005.png,Hernia,4,3,78,F,PA,2686,2991,0.143,0.143
9,00000003_006.png,Hernia,5,3,79,F,PA,2992,2991,0.143,0.143


In [7]:
import pandas as pd

dados_dicionario = {
    "Coluna": [
        "Image Index", "Finding Labels", "Follow-up #", "Patient ID", 
        "Patient Age", "Patient Sex", "View Position", "OriginalImage[Width", 
        "Height]", "OriginalImagePixelSpacing[x", "y]"
    ],
    "Significado_no_Dataset": [
        "Nome exato do arquivo da imagem (ex: 00000001_000.png).",
        "Doenças encontradas separadas por '|' ou 'No Finding'.",
        "Número do exame na linha do tempo do paciente.",
        "Código único que identifica o paciente.",
        "Idade do paciente no momento do raio-X.",
        "Sexo biológico do paciente (M/F).",
        "Posição do disparo do raio-X (PA ou AP).",
        "Largura da imagem original em pixels.",
        "Altura da imagem original em pixels.",
        "Distância física entre pixels no eixo X (mm).",
        "Distância física entre pixels no eixo Y (mm)."
    ],
    "Relevancia_para_o_Projeto": [
        "Chave primária para carregar a imagem.",
        "Target (Y) que o modelo vai prever.",
        "Útil para analisar evolução temporal.",
        "Evita data leakage na separação treino/teste.",
        "Feature extra opcional.",
        "Feature extra (análise de viés).",
        "Diferencia exames de rotina de UTI.",
        "Informativo para redimensionamento.",
        "Informativo para redimensionamento.",
        "Cálculos de área do mundo real.",
        "Cálculos de área do mundo real."
    ]
}

df_dicionario = pd.DataFrame(dados_dicionario)

In [8]:
df_dicionario

,Coluna,Significado_no_Dataset,Relevancia_para_o_Projeto
0,Image Index,Nome exato do arquivo da imagem (ex: 00000001_...,Chave primária para carregar a imagem.
1,Finding Labels,Doenças encontradas separadas por '|' ou 'No F...,Target (Y) que o modelo vai prever.
2,Follow-up #,Número do exame na linha do tempo do paciente.,Útil para analisar evolução temporal.
3,Patient ID,Código único que identifica o paciente.,Evita data leakage na separação treino/teste.
4,Patient Age,Idade do paciente no momento do raio-X.,Feature extra opcional.
5,Patient Sex,Sexo biológico do paciente (M/F).,Feature extra (análise de viés).
6,View Position,Posição do disparo do raio-X (PA ou AP).,Diferencia exames de rotina de UTI.
7,OriginalImage[Width,Largura da imagem original em pixels.,Informativo para redimensionamento.
8,Height],Altura da imagem original em pixels.,Informativo para redimensionamento.
9,OriginalImagePixelSpacing[x,Distância física entre pixels no eixo X (mm).,Cálculos de área do mundo real.


## EDA

coisas que eu faria so de olhar. 
renomear colunas e padronizar nomes delas, 
separar achados quando são mais de um em ahca dois 3 e etc 
identificar as diferenças entre quantidade de doenças achadas  sexo dos pacientes 
peaquisar melhor o que é view position



Follow-up # é quantos exames o paciente já tem alem desse, é acumulativo




como o fato do mesmo paciente fazer varios exames influencia a base de dados?

In [23]:
df[df["Patient Age"] < 18]

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Sex,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
2031,00000522_000.png,Atelectasis|Infiltration,0,522,17,M,AP,2048,2500,0.171000,0.171000
2146,00000564_000.png,No Finding,0,564,16,M,PA,2048,2500,0.171000,0.171000
2652,00000702_000.png,No Finding,0,702,16,F,PA,2048,2500,0.171000,0.171000
2863,00000763_000.png,No Finding,0,763,16,M,PA,2048,2500,0.168000,0.168000
3067,00000813_001.png,No Finding,0,813,16,F,PA,2048,2500,0.168000,0.168000
...,...,...,...,...,...,...,...,...,...,...,...
112076,00030773_000.png,No Finding,0,30773,9,M,PA,2021,2021,0.194311,0.194311
112077,00030773_001.png,No Finding,1,30773,9,M,PA,2021,2021,0.194311,0.194311
112083,00030779_000.png,No Finding,0,30779,17,M,PA,2021,2021,0.194311,0.194311
112104,00030792_000.png,No Finding,0,30792,9,F,PA,1775,1712,0.194311,0.194311
